# Tokenizadores — BPE vs WordPiece vs SentencePiece

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

Tokenizadores subword dividem texto em pedaços que balanceiam tamanho de vocabulário e tratamento de OOV. BPE mescla iterativamente o par adjacente mais frequente; WordPiece escolhe o merge que maximiza a verossimilhança do corpus; SentencePiece opera direto em bytes/UTF-8 (sem assumir espaços).


## Formulação Matemática

BPE escolhe merges $(a, b) \to ab$ que maximizam a frequência:

$$(a^*, b^*) = \arg\max_{(a, b)} \text{count}(ab \mid \text{corpus})$$

WordPiece escolhe merges que maximizam o *score*

$$\text{score}(a, b) = \frac{\text{count}(ab)}{\text{count}(a)\cdot \text{count}(b)}$$


## Implementação


In [ ]:
# pip install tokenizers
from tokenizers import Tokenizer
from tokenizers.models import BPE, WordPiece
from tokenizers.trainers import BpeTrainer, WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace


In [ ]:
corpus = [
    'the quick brown fox jumps over the lazy dog',
    'jumping cats and lazy dogs',
    'transformers tokenise subwords efficiently',
] * 50  # tiny corpus, repeated

def train_bpe(corpus, vocab_size=80):
    tok = Tokenizer(BPE(unk_token='[UNK]'))
    tok.pre_tokenizer = Whitespace()
    trainer = BpeTrainer(vocab_size=vocab_size, special_tokens=['[UNK]'])
    tok.train_from_iterator(corpus, trainer)
    return tok

def train_wp(corpus, vocab_size=80):
    tok = Tokenizer(WordPiece(unk_token='[UNK]'))
    tok.pre_tokenizer = Whitespace()
    trainer = WordPieceTrainer(vocab_size=vocab_size, special_tokens=['[UNK]'])
    tok.train_from_iterator(corpus, trainer)
    return tok


## Experimento


In [ ]:
bpe = train_bpe(corpus)
wp = train_wp(corpus)

sentence = 'jumping subwords tokenise efficiently'
print('BPE      :', bpe.encode(sentence).tokens)
print('WordPiece:', wp.encode(sentence).tokens)


## Discussão

- BPE é guloso por frequência; o score do WordPiece é mais próximo de critério de verossimilhança e trata morfologia um pouco melhor.
- SentencePiece (Unigram ou BPE) opera em byte level e é agnóstico a espaços — preferido para não-inglês / multilíngue.
- O tamanho do vocab é hiperparâmetro: 16k–64k para monolíngue, 200k+ para multilíngue.


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
